# Widget check

Something in this folder draws a widget and nothing appears, or a spinner turns
and never stops. This notebook finds out which part is at fault, so that the
answer is one message rather than a week of guessing.

Do **Kernel -> Restart** first, so that nothing left over from earlier confuses
it. Then run the cells below **one at a time, from the top**, and write down the
number of the first one that does not do what its heading says. Do not skip
ahead: which cell fails first is the whole of the answer, and a later cell
cannot tell you anything the first failure has not already settled.

Send that number, and the output of cell 1 and cell 6, to your instructor.

## 1. Which Python is running this notebook

Expected: a path with `.pixi` in it, and `True` on the last two lines.

In [ ]:
import sys, platform, pathlib
print("python     ", sys.version.split()[0], platform.machine())
print("prefix     ", sys.prefix)
import ipywidgets, anywidget, ipykernel
print("ipywidgets ", ipywidgets.__version__)
print("anywidget  ", anywidget.__version__)
print("ipykernel  ", ipykernel.__version__)
print()
print("running in the course environment:", ".pixi" in sys.prefix)
frontend = pathlib.Path(sys.prefix, "share/jupyter/nbextensions/anywidget/index.js")
print("widget support file present:      ", frontend.is_file())
print("  ", frontend)

## 2. A widget VS Code already has

Expected: a **slider you can drag**.

This one is built into VS Code and needs nothing fetched from anywhere. If it
fails, the fault is in the editor or in the connection to the kernel, and has
nothing to do with the course widgets.

In [ ]:
import ipywidgets
ipywidgets.IntSlider(value=5, min=0, max=10, description="drag me")

## 3. A widget VS Code has to fetch support for

Expected: a green box reading **custom widget loaded**.

This is the smallest widget of the same *kind* as `%%steps`. Unlike the slider,
VS Code has to obtain a piece of supporting code before it can draw it. If the
slider worked and this does not, obtaining that code is the problem, and cell 6
below is the setting that decides where VS Code looks for it.

In [ ]:
import anywidget
class Probe(anywidget.AnyWidget):
    _esm = '''
    export default { render({ el }) {
        el.style.cssText = "padding:8px;border:2px solid green;font:14px system-ui";
        el.textContent = "custom widget loaded";
    } }
    '''
Probe()

## 4. The steps widget

Expected: a card headed **Line 3**, listing the substitution and reduction steps
of `x * y + 4`.

In [ ]:
import steps_widget

In [ ]:
%%steps
x = 7
y = 5
z = x * y + 4 # PRINT STEPS

## 5. The steps widget a second time

Expected: a second card, headed **Line 2**.

Run this one even if cell 4 already failed, because a second widget can fail
differently from the first, and the difference is informative. Say which of the
three happened: a card as expected, nothing at all, or a spinner that turns
without ever stopping.

In [ ]:
%%steps
a = 3
b = a * 10 - 5 # PRINT STEPS

## 6. Where VS Code looks for widget support

Expected: a line about `jupyter.widgetScriptSources`. Whatever it prints, send
it along with the number of the first cell that failed.

In [ ]:
import json, os, pathlib, re, sys

if sys.platform == "darwin":
    user_settings = pathlib.Path.home() / "Library/Application Support/Code/User/settings.json"
elif os.name == "nt":
    user_settings = pathlib.Path(os.environ.get("APPDATA", "")) / "Code/User/settings.json"
else:
    user_settings = pathlib.Path.home() / ".config/Code/User/settings.json"

def read(path):
    """Load a settings file, which is JSON with comments and trailing commas."""
    text = re.sub(r"//[^\n]*", "", path.read_text(encoding="utf-8"))
    return json.loads(re.sub(r",(\s*[}\]])", r"\1", text))

for label, path in [("your VS Code", user_settings),
                    ("this folder ", pathlib.Path.cwd() / ".vscode/settings.json")]:
    if not path.is_file():
        print(f"{label}: no settings file at {path}")
        continue
    try:
        settings = read(path)
    except Exception as error:
        print(f"{label}: could not be read ({error})")
        continue
    key = "jupyter.widgetScriptSources"
    print(f"{label}: {key} = {settings.get(key, '(not set)')}")

## 7. If a widget failed, the editor's own log says why

VS Code keeps a log of what it did while trying to draw a widget, and it
records which of the two places it looked in, and what it found there. That log
is the one thing that settles this, so send it even if nothing above looked
unusual.

Open it from the menu: **View -> Output**, then choose **Jupyter** in the
dropdown at the top right of the panel that appears. Copy everything from the
last time you ran a cell. The lines that matter say `Widget` or `nbextensions`
in them, and look something like these:

    Script source for Widget anywidget@~0.11.* not found in local, cdn
    Widget Script Source not found for anywidget@~0.11.* from local
    No nbextensions folder found for kernel ...
    Widget load failure ...

If there is nothing about widgets in the log at all, that is itself the answer,
and worth saying: it means the request to fetch the widget code was started and
never came back, rather than failing.

If you are asked for more detail, set the log to its most talkative setting
first and run the cells again: **Cmd/Ctrl+Shift+P**, type `Preferences: Open
Settings (UI)`, search for `jupyter.logging.level`, and set it to `trace`.